In [18]:
# =====================================================================
# GenePT kNN benchmark - all conditions, dense log2FC target, full baselines
# =====================================================================
import os, pickle, numpy as np, pandas as pd
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import KFold
from scipy.stats import pearsonr

EMB_PATH    = "/content/genept/GenePT_emebdding_v2/GenePT_gene_protein_embedding_model_3_text.pickle"
PERTURB_DIR = "/content/drive/MyDrive/benchmarking_paper/datasets/perturbnet_outputs"
CONDITIONS  = ["Rest", "Stim8hr", "Stim48hr"]
KS          = [3, 5, 10, 15, 25]
WEIGHTS     = ["uniform", "distance"]
N_RANDOM_SEEDS, N_SPLITS, SEED = 5, 5, 42
OUT_CSV     = "/content/genept/knn_benchmark_all_conditions.csv"

# Mount Drive if needed
if not os.path.isdir(PERTURB_DIR):
    from google.colab import drive
    drive.mount("/content/drive")

# Load embeddings once
if "genept" not in globals():
    print("Loading GenePT embeddings...")
    with open(EMB_PATH, "rb") as f:
        genept = pickle.load(f)
    genept = {str(k).upper(): np.asarray(v, np.float64) for k, v in genept.items()}
    print(f"  {len(genept)} genes, dim {next(iter(genept.values())).shape[0]}")

def per_tf_pearson(Yt, Yp):
    return np.array([pearsonr(yt, yp)[0] if yt.std() > 1e-12 and yp.std() > 1e-12 else np.nan
                     for yt, yp in zip(Yt, Yp)])

def cv_knn(Z, Y, k, weights):
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED); rs = []
    for tr, te in kf.split(Z):
        m = KNeighborsRegressor(n_neighbors=k, weights=weights, metric="euclidean").fit(Z[tr], Y[tr])
        rs.append(per_tf_pearson(Y[te], m.predict(Z[te])))
    rs = np.concatenate(rs); return np.nanmean(rs), np.nanstd(rs)

def cv_train_mean(Y):
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED); rs = []
    for tr, te in kf.split(Y):
        mean_resp = np.repeat(Y[tr].mean(0, keepdims=True), len(te), axis=0)
        rs.append(per_tf_pearson(Y[te], mean_resp))
    rs = np.concatenate(rs); return np.nanmean(rs), np.nanstd(rs)

def cv_random(Y, shape, k):
    vals = []
    for s in range(N_RANDOM_SEEDS):
        rng = np.random.default_rng(s)
        Zr = rng.normal(size=shape); Zr /= np.linalg.norm(Zr, axis=1, keepdims=True)
        vals.append(cv_knn(Zr, Y, k, "uniform")[0])
    return np.mean(vals), np.std(vals)

all_rows, summary = [], []
for cond in CONDITIONS:
    fp = f"{PERTURB_DIR}/perturbnet_{cond}.csv"
    if not os.path.exists(fp):
        print(f"SKIP {cond}: file not found"); continue
    print(f"\n=== {cond} ===  (reading large file...)")
    e = pd.read_csv(fp, usecols=["source_TF", "target_gene", "log2FC"])
    e["source_TF"]   = e["source_TF"].astype(str).str.upper().str.strip()
    e["target_gene"] = e["target_gene"].astype(str).str.upper().str.strip()
    Y = e.pivot_table(index="source_TF", columns="target_gene",
                      values="log2FC", aggfunc="mean").fillna(0.0)

    tfs = [t for t in Y.index if t in genept]
    Z = np.vstack([genept[t] for t in tfs]); Z /= np.linalg.norm(Z, axis=1, keepdims=True)
    Yv = Y.loc[tfs].values
    print(f"  {len(tfs)} TFs x {Yv.shape[1]} genes ({len(Y.index)-len(tfs)} dropped, no embedding)")

    best = (-1, None)
    for k in KS:
        for w in WEIGHTS:
            m, s = cv_knn(Z, Yv, k, w)
            all_rows.append([cond, f"kNN k={k} ({w})", m, s])
            if m > best[0]: best = (m, k)
    tm, tms = cv_train_mean(Yv)
    all_rows.append([cond, "Train Mean", tm, tms])
    for k in KS:
        rm, rs = cv_random(Yv, Z.shape, k)
        all_rows.append([cond, f"Random k={k}", rm, rs])

    rand_at_best = next(r for r in all_rows if r[0]==cond and r[1]==f"Random k={best[1]}")[2]
    summary.append([cond, round(best[0],3), best[1], round(tm,3), round(rand_at_best,3),
                    round(best[0]-rand_at_best,3), "YES" if best[0] > tm else "NO"])
    print(f"  best kNN (k={best[1]}): {best[0]:.3f} | Train Mean: {tm:.3f} | "
          f"Random@k: {rand_at_best:.3f} | beats baseline: {'YES' if best[0] > tm else 'NO'}")

pd.DataFrame(all_rows, columns=["condition","model","pearson_mean","pearson_std"]).to_csv(OUT_CSV, index=False)
summary_df = pd.DataFrame(summary, columns=["condition","best_kNN","best_k","Train_Mean",
                                            "Random_at_best_k","lift_vs_random","kNN_beats_TrainMean"])
print("\n" + "="*70 + "\nSUMMARY\n" + "="*70)
print(summary_df.to_string(index=False))
print(f"\nFull results saved to: {OUT_CSV}")


=== Rest ===  (reading large file...)
  560 TFs x 10282 genes (1 dropped, no embedding)
  best kNN (k=25): 0.118 | Train Mean: 0.168 | Random@k: 0.112 | beats baseline: NO

=== Stim8hr ===  (reading large file...)
  595 TFs x 10282 genes (1 dropped, no embedding)
  best kNN (k=25): 0.137 | Train Mean: 0.189 | Random@k: 0.134 | beats baseline: NO

=== Stim48hr ===  (reading large file...)
  575 TFs x 10282 genes (1 dropped, no embedding)
  best kNN (k=25): 0.109 | Train Mean: 0.160 | Random@k: 0.105 | beats baseline: NO

SUMMARY
condition  best_kNN  best_k  Train_Mean  Random_at_best_k  lift_vs_random kNN_beats_TrainMean
     Rest     0.118      25       0.168             0.112           0.006                  NO
  Stim8hr     0.137      25       0.189             0.134           0.004                  NO
 Stim48hr     0.109      25       0.160             0.105           0.004                  NO

Full results saved to: /content/genept/knn_benchmark_all_conditions.csv


In [6]:
import pandas as pd, numpy as np, json, os

# --- Your TF universe: the Lambert 1,639 human TFs ---
# Point this at your Lambert list. It just needs a column of gene symbols.
LAMBERT_PATH = "/content/drive/MyDrive/benchmarking_paper/datasets/tf_lists/humanTFs_1639_clean.csv"   # <-- adjust to your file
lambert = pd.read_csv(LAMBERT_PATH)
tf_symbols = lambert.iloc[:, 0].astype(str).str.upper().tolist()   # first column = symbols
print(f"Lambert TFs listed: {len(tf_symbols)}")

# --- Keep only TFs GenePT covers; stack their vectors into Z ---
covered, missing, vecs = [], [], []
for tf in tf_symbols:
    if tf in genept:
        covered.append(tf)
        vecs.append(genept[tf])
    else:
        missing.append(tf)

Z = np.vstack(vecs)
print(f"Covered (have embedding): {len(covered)}")
print(f"Missing (no embedding):   {len(missing)}  ->  {missing[:10]}")
print(f"Z shape: {Z.shape}")

# --- Save the three artifacts (match Task 05 / 06 filenames) ---
OUT = "/content/genept/embeddings"
os.makedirs(OUT, exist_ok=True)
np.save(f"{OUT}/genept_ncbi_uniprot_Z.npy", Z)
pd.Series(covered, name="TF").to_csv(f"{OUT}/Z_row_order.csv", index=False)
json.dump({"covered": covered, "missing": missing}, open(f"{OUT}/coverage_manifest.json", "w"))
print("Saved: Z.npy, Z_row_order.csv, coverage_manifest.json")

FileNotFoundError: [Errno 2] No such file or directory: '/content/lambert_tfs.csv'

In [7]:
LAMBERT_PATH = "/content/drive/MyDrive/benchmarking_paper/datasets/tf_lists/humanTFs_1639_clean.csv"   # <-- adjust to your file

# 1. Raw first 5 lines (shows the true delimiter and structure, no pandas guessing)
print("=== RAW LINES ===")
with open(LAMBERT_PATH) as f:
    for i, line in enumerate(f):
        print(repr(line))
        if i >= 4:
            break

=== RAW LINES ===
'gene_id,gene_symbol\n'
'ENSG00000267179,AC008770.3\n'
'ENSG00000267281,AC023509.3\n'
'ENSG00000233757,AC092835.1\n'
'ENSG00000264668,AC138696.1\n'


In [10]:
import pandas as pd, numpy as np, json, os

LAMBERT_PATH = "/content/drive/MyDrive/benchmarking_paper/datasets/tf_lists/humanTFs_1639_clean.csv"   # <-- adjust to your file
tf_symbols = lambert["gene_symbol"].astype(str).str.upper().str.strip().tolist()
print(f"Lambert TFs: {len(tf_symbols)}")

seen, covered, missing, vecs = set(), [], [], []
for tf in tf_symbols:
    if tf in seen:
        continue
    seen.add(tf)
    if tf in genept:
        covered.append(tf); vecs.append(genept[tf])
    else:
        missing.append(tf)

Z = np.vstack(vecs)
print(f"Covered: {len(covered)} | Missing: {len(missing)}")
print("first 15 missing:", missing[:15])
print("Z shape:", Z.shape)

OUT = "/content/genept/embeddings"; os.makedirs(OUT, exist_ok=True)
np.save(f"{OUT}/genept_ncbi_uniprot_Z.npy", Z)
pd.Series(covered, name="TF").to_csv(f"{OUT}/Z_row_order.csv", index=False)
json.dump({"covered": covered, "missing": missing}, open(f"{OUT}/coverage_manifest.json", "w"))
print("Saved: Z.npy, Z_row_order.csv, coverage_manifest.json")

Lambert TFs: 1639
Covered: 1623 | Missing: 16
first 15 missing: ['AC008770.3', 'AC023509.3', 'AC092835.1', 'AC138696.1', 'ARNTL', 'BORCS8-MEF2B', 'C11ORF95', 'CENPBD1', 'CTCFL', 'DUX1', 'DUX3', 'RFX3', 'ZBED9', 'ZNF705E', 'ZNF781']
Z shape: (1623, 3072)
Saved: Z.npy, Z_row_order.csv, coverage_manifest.json


In [12]:
import pandas as pd
import numpy as np

PERTURB_DIR = "/content/drive/MyDrive/benchmarking_paper/datasets/perturbnet_outputs"

def build_Y(cond):
    edges = pd.read_csv(f"{PERTURB_DIR}/perturb_net_tfs_only_{cond}_filtered.csv")
    edges["source_TF"]   = edges["source_TF"].astype(str).str.upper().str.strip()
    edges["target_gene"] = edges["target_gene"].astype(str).str.upper().str.strip()
    # rows = perturbed TF, cols = target gene, values = log2FC, missing pair = 0
    Y = edges.pivot_table(index="source_TF", columns="target_gene",
                          values="log2FC", aggfunc="mean", fill_value=0.0)
    return Y

Y_rest = build_Y("Rest")
print("Y_rest shape (TFs x genes):", Y_rest.shape)
print("n perturbed TFs:", Y_rest.shape[0])
print("n target genes :", Y_rest.shape[1])
print(Y_rest.iloc[:3, :5])

Y_rest shape (TFs x genes): (409, 8288)
n perturbed TFs: 409
n target genes : 8288
target_gene  AAAS  AACS  AAGAB  AAK1  AAMP
source_TF                                 
ADNP          0.0   0.0    0.0   0.0   0.0
ADNP2         0.0   0.0    0.0   0.0   0.0
AEBP2         0.0   0.0    0.0   0.0   0.0


In [13]:
import numpy as np

rest_tfs = Y_rest.index.tolist()
in_emb = [t for t in rest_tfs if t in genept]
no_emb = [t for t in rest_tfs if t not in genept]

print(f"Rest TFs: {len(rest_tfs)} | with embedding: {len(in_emb)} | without: {len(no_emb)}")
print("dropped (no embedding):", no_emb)

# Aligned matrices, same TF order
Z_rest = np.vstack([genept[t] for t in in_emb])
Y_rest_aligned = Y_rest.loc[in_emb]

# L2-normalize embeddings so Euclidean kNN behaves like cosine
Z_rest = Z_rest / np.linalg.norm(Z_rest, axis=1, keepdims=True)

print("Z_rest:", Z_rest.shape, "| Y_rest_aligned:", Y_rest_aligned.shape)

Rest TFs: 409 | with embedding: 409 | without: 0
dropped (no embedding): []
Z_rest: (409, 3072) | Y_rest_aligned: (409, 8288)


In [14]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import KFold
from scipy.stats import pearsonr
import numpy as np, pandas as pd

Zr = Z_rest
Yr = Y_rest_aligned.values

def per_tf_pearson(y_true_mat, y_pred_mat):
    rs = []
    for yt, yp in zip(y_true_mat, y_pred_mat):
        rs.append(pearsonr(yt, yp)[0] if yt.std() > 1e-12 and yp.std() > 1e-12 else np.nan)
    return np.array(rs)

def cv_knn(Z, Y, k, weights, n_splits=5, seed=42):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    rs = []
    for tr, te in kf.split(Z):
        m = KNeighborsRegressor(n_neighbors=k, weights=weights, metric="euclidean")
        m.fit(Z[tr], Y[tr])
        rs.append(per_tf_pearson(Y[te], m.predict(Z[te])))
    rs = np.concatenate(rs)
    return np.nanmean(rs), np.nanstd(rs)

def cv_train_mean(Y, n_splits=5, seed=42):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    rs = []
    for tr, te in kf.split(Y):
        mean_resp = np.repeat(Y[tr].mean(axis=0, keepdims=True), len(te), axis=0)
        rs.append(per_tf_pearson(Y[te], mean_resp))
    rs = np.concatenate(rs)
    return np.nanmean(rs), np.nanstd(rs)

rows = []
for k in [3, 5, 10, 15, 25]:
    for w in ["uniform", "distance"]:
        m, s = cv_knn(Zr, Yr, k, w)
        rows.append({"model": f"kNN k={k} ({w})", "pearson_mean": m, "pearson_std": s})

bm, bs = cv_train_mean(Yr)
rows.append({"model": "Train Mean (baseline)", "pearson_mean": bm, "pearson_std": bs})

rng = np.random.default_rng(0)
Z_rand = rng.normal(size=Zr.shape)
Z_rand = Z_rand / np.linalg.norm(Z_rand, axis=1, keepdims=True)
rm, rsd = cv_knn(Z_rand, Yr, 10, "distance")
rows.append({"model": "Random Emb (k=10)", "pearson_mean": rm, "pearson_std": rsd})

res = pd.DataFrame(rows).sort_values("pearson_mean", ascending=False).reset_index(drop=True)
print(res.to_string(index=False))

                model  pearson_mean  pearson_std
Train Mean (baseline)      0.030291     0.044603
  kNN k=25 (distance)      0.017242     0.043809
   kNN k=25 (uniform)      0.017173     0.043386
   kNN k=15 (uniform)      0.014635     0.045385
  kNN k=15 (distance)      0.014509     0.045381
  kNN k=10 (distance)      0.013230     0.046939
   kNN k=10 (uniform)      0.013189     0.046883
    kNN k=5 (uniform)      0.008676     0.044476
   kNN k=5 (distance)      0.008648     0.043996
    kNN k=3 (uniform)      0.005307     0.031421
   kNN k=3 (distance)      0.005233     0.031066
    Random Emb (k=10)      0.004137     0.024797


In [15]:
import pandas as pd
PERTURB_DIR = "/content/drive/MyDrive/benchmarking_paper/datasets/perturbnet_outputs"
d = pd.read_csv(f"{PERTURB_DIR}/perturbnet_Rest.csv")
print("shape:", d.shape)
print("columns:", d.columns.tolist())
print(d.head())

shape: (5767641, 8)
columns: ['source_TF', 'target_gene', 'log2FC', 'adj_p_value', 'p_value', 'zscore', 'culture_condition', 'n_guides_averaged']
  source_TF target_gene    log2FC  adj_p_value   p_value    zscore  \
0      ADNP        DPM1 -0.053887     0.999990  0.407165 -0.828893   
1      ADNP       SCYL3 -0.176545     0.792338  0.049566 -1.963690   
2      ADNP    C1orf112  0.107363     0.999990  0.382820  0.872712   
3      ADNP         CFH  0.399823     0.999990  0.621335  0.493959   
4      ADNP       FUCA2  0.082811     0.999990  0.609365  0.510981   

  culture_condition  n_guides_averaged  
0              Rest                  1  
1              Rest                  1  
2              Rest                  1  
3              Rest                  1  
4              Rest                  1  


In [16]:
import pandas as pd, numpy as np

PERTURB_DIR = "/content/drive/MyDrive/benchmarking_paper/datasets/perturbnet_outputs"

def build_Y_dense(cond):
    e = pd.read_csv(f"{PERTURB_DIR}/perturbnet_{cond}.csv",
                    usecols=["source_TF", "target_gene", "log2FC"])
    e["source_TF"]   = e["source_TF"].astype(str).str.upper().str.strip()
    e["target_gene"] = e["target_gene"].astype(str).str.upper().str.strip()
    return e.pivot_table(index="source_TF", columns="target_gene",
                         values="log2FC", aggfunc="mean")

print("Reading dense Rest matrix (big file, give it a minute)...")
Yd = build_Y_dense("Rest").fillna(0.0)
print("Dense Y_rest shape:", Yd.shape)

# align + rerun the same CV
tfs = [t for t in Yd.index if t in genept]
Z = np.vstack([genept[t] for t in tfs]); Z = Z / np.linalg.norm(Z, axis=1, keepdims=True)
Yv = Yd.loc[tfs].values
print(f"Aligned: {len(tfs)} TFs x {Yv.shape[1]} genes")

rows = []
for k in [3, 5, 10, 15, 25]:
    for w in ["uniform", "distance"]:
        m, s = cv_knn(Z, Yv, k, w)
        rows.append({"model": f"kNN k={k} ({w})", "pearson_mean": m, "pearson_std": s})
bm, bs = cv_train_mean(Yv); rows.append({"model": "Train Mean", "pearson_mean": bm, "pearson_std": bs})
rng = np.random.default_rng(0); Zr = rng.normal(size=Z.shape); Zr /= np.linalg.norm(Zr, axis=1, keepdims=True)
rm, rsd = cv_knn(Zr, Yv, 10, "distance"); rows.append({"model": "Random Emb", "pearson_mean": rm, "pearson_std": rsd})

print(pd.DataFrame(rows).sort_values("pearson_mean", ascending=False).to_string(index=False))

Reading dense Rest matrix (big file, give it a minute)...
Dense Y_rest shape: (561, 10282)
Aligned: 560 TFs x 10282 genes
              model  pearson_mean  pearson_std
         Train Mean      0.167855     0.101946
 kNN k=25 (uniform)      0.118303     0.079311
kNN k=25 (distance)      0.116528     0.079033
 kNN k=15 (uniform)      0.097603     0.073525
kNN k=15 (distance)      0.096022     0.073464
 kNN k=10 (uniform)      0.083917     0.071174
kNN k=10 (distance)      0.082336     0.071011
         Random Emb      0.080153     0.071895
  kNN k=5 (uniform)      0.060661     0.067721
 kNN k=5 (distance)      0.060070     0.067550
  kNN k=3 (uniform)      0.050097     0.066156
 kNN k=3 (distance)      0.049909     0.066049


In [17]:
def cv_knn_random(Y, Z_shape, k, n_seeds=5):
    vals = []
    for s in range(n_seeds):
        rng = np.random.default_rng(s)
        Zr = rng.normal(size=Z_shape); Zr /= np.linalg.norm(Zr, axis=1, keepdims=True)
        vals.append(cv_knn(Zr, Y, k, "uniform")[0])
    return np.mean(vals), np.std(vals)

comp = []
for k in [3, 5, 10, 15, 25]:
    g = cv_knn(Z, Yv, k, "uniform")[0]
    rmean, rstd = cv_knn_random(Yv, Z.shape, k)
    comp.append({"k": k, "GenePT": round(g, 3), "Random": round(rmean, 3),
                 "Random_std": round(rstd, 3), "lift": round(g - rmean, 3)})
print(pd.DataFrame(comp).to_string(index=False))
print(f"\nTrain Mean ceiling: {cv_train_mean(Yv)[0]:.3f}")

 k  GenePT  Random  Random_std  lift
 3   0.050   0.049       0.004 0.002
 5   0.061   0.060       0.003 0.000
10   0.084   0.080       0.002 0.004
15   0.098   0.094       0.003 0.003
25   0.118   0.112       0.003 0.006

Train Mean ceiling: 0.168
